# Cats vs Dogs Image Classification

An end-to-end computer-vision experiment that trains a compact convolutional neural network (CNN) to distinguish cats from dogs.

## Project snapshot

| | |
|---|---|
| **Goal** | Binary image classification: cat (`0`) or dog (`1`) |
| **Data** | Kaggle *Microsoft Cats vs Dogs* images, with corrupt files removed |
| **Approach** | Stratified 80/10/10 split, `tf.data` pipeline, three-block CNN |
| **Evaluation** | Binary cross-entropy and accuracy on a held-out test set |
| **Status** | Experimental workflow implemented; no training results are saved in this notebook |

The workflow covers data acquisition, validation, split construction, input processing, model training, and final evaluation.


## Run requirements

Open the notebook from its project folder, or from a Jupyter server launched anywhere inside this repository, with KaggleHub access configured. The download and full image scan require an internet connection and local storage.

- The dataset download uses `kagglehub`; Kaggle credentials may be required.
- Pixel values are normalized once in the input pipeline before they reach the model.
- Training is compute-intensive and is best run with GPU acceleration.
- Expect this notebook to take longer than the smaller tabular projects in the portfolio.


In [ ]:
# --- Imports ---
import os

import kagglehub
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

# --- Configuration (kept consistent across the notebook) ---
IMG_SIZE = (150, 150)      # (height, width)
BATCH_SIZE = 32
EPOCHS = 10


## 1. Download and locate the dataset

In [ ]:
# Download latest version of the dataset (this returns a local folder path)
path = kagglehub.dataset_download("shaunthesheep/microsoft-catsvsdogs-dataset")
print("Path to dataset files:", path)

# The dataset structure is usually: <path>/PetImages/{Cat,Dog}/...
pet_images_dir = os.path.join(path, "PetImages")
cat_dir = os.path.join(pet_images_dir, "Cat")
dog_dir = os.path.join(pet_images_dir, "Dog")

# Optional sanity check: list top-level entries
os.listdir(pet_images_dir)


## 2. Collect file paths

In [ ]:
# Collect absolute paths to each image file
list_of_cat_file_paths = [os.path.join(cat_dir, f) for f in os.listdir(cat_dir)]
list_of_dog_file_paths = [os.path.join(dog_dir, f) for f in os.listdir(dog_dir)]

print("Raw cat images:", len(list_of_cat_file_paths))
print("Raw dog images:", len(list_of_dog_file_paths))


## 3. Validate image files

In [ ]:
def is_valid_image_tf(path: str) -> bool:
    """Return True if TensorFlow can read + decode the file as an image.

    The Cats vs Dogs dataset is known to contain a small number of corrupt files.
    We filter them out up front to avoid training crashes later.
    """
    try:
        img_bytes = tf.io.read_file(path)
        # decode_image supports JPEG/PNG/etc, and returns a 3-channel tensor when channels=3
        img = tf.io.decode_image(img_bytes, channels=3, expand_animations=False)
        # Resize here so decode + resize happens during validation too (keeps original behavior)
        _ = tf.image.resize(img, IMG_SIZE)
        return True
    except tf.errors.InvalidArgumentError:
        return False

# This can take a little while because it reads every file once.
list_of_dog_file_paths = [p for p in list_of_dog_file_paths if is_valid_image_tf(p)]
list_of_cat_file_paths = [p for p in list_of_cat_file_paths if is_valid_image_tf(p)]

print("Valid cat images:", len(list_of_cat_file_paths))
print("Valid dog images:", len(list_of_dog_file_paths))


## 4. Create labels and train/validation/test splits

In [ ]:
# Create a DataFrame with file paths and labels (0 for cat, 1 for dog)
df = pd.DataFrame({
    "file_path": list_of_cat_file_paths + list_of_dog_file_paths,
    "label": [0] * len(list_of_cat_file_paths) + [1] * len(list_of_dog_file_paths),
})

# 80% train, 20% temp (later split into val/test)
X_train, X_temp, y_train, y_temp = train_test_split(
    df["file_path"],
    df["label"],
    test_size=0.2,
    stratify=df["label"],   # preserve class balance
    random_state=42,
)

# Split temp into 50% validation, 50% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42,
)

print("Train:", len(X_train), " Val:", len(X_val), " Test:", len(X_test))
print("Train label mean (dogs=1):", float(y_train.mean()))
print("Val label mean (dogs=1):", float(y_val.mean()))
print("Test label mean (dogs=1):", float(y_test.mean()))


## 5. Build the `tf.data` input pipeline

In [ ]:
def load_image(path, label):
    """Read an image from disk and return (image_tensor, label).

    - Reads bytes from a file path
    - Decodes to an RGB tensor
    - Resizes to IMG_SIZE
    - Scales to [0, 1] float32
    """
    img_bytes = tf.io.read_file(path)
    img = tf.io.decode_image(img_bytes, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0

    label = tf.cast(label, tf.float32)
    return img, label

# Training dataset
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(min(len(X_train), 10000), seed=42)
train_ds = train_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Validation dataset
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_ds = val_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Test dataset
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))
test_ds = test_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


## 6. Define the CNN

In [ ]:
# A simple CNN with 3 convolution blocks + global average pooling.
# Output is a single sigmoid unit for binary classification (cat vs dog).
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(*IMG_SIZE, 3)),
    tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, 3, activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(),

    # GlobalAveragePooling2D replaces Flatten and reduces parameters.
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(1, activation="sigmoid"),
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()


## 7. Train the model

In [ ]:
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
)


## 8. Evaluate on the held-out test set

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)


## Results and takeaways

This notebook intentionally ships without saved training output because the result depends on downloading the image corpus and running a compute-intensive training job. After execution, the held-out test loss and accuracy printed above are the decision metrics; validation performance should be used during iteration, while the test split should remain untouched until final evaluation.

The implementation demonstrates a reproducible image pipeline, explicit corrupt-file handling, stratified data splits, single-pass normalization, and a compact CNN with dropout.


## Limitations and next steps

- Accuracy alone can hide class-specific errors; add a confusion matrix, precision, recall, and per-class examples.
- The model has no augmentation or transfer learning, so it may be sensitive to pose, background, and lighting.
- Compare this baseline with a pretrained backbone and document the accuracy/latency trade-off.
- Add early stopping and save the best validation checkpoint before final test evaluation.
